# DeCad - train the schematic text reader (Colab T4)

Trains `training/schematic_ocr.py` (compact CRNN for pin numbers, net labels, part numbers) for longer than a 4 GB laptop GPU allows, then saves `model.pt` + `config.json`.

1. Runtime -> Change runtime type -> **T4 GPU**.
2. Upload `schematic_ocr.py` (and optionally `real-crops.zip` made by `tools/harvest_crops.py`) when asked.
3. Download `schematic-ocr.zip` at the end and unzip it into `checkpoints/schematic-ocr/` of the app.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'
!pip -q install opencv-python-headless

In [ ]:
from google.colab import files
up = files.upload()   # schematic_ocr.py [+ real-crops.zip]
import zipfile, os
if 'real-crops.zip' in up: zipfile.ZipFile('real-crops.zip').extractall('real')
REAL = '--real real/crops.jsonl' if os.path.exists('real/crops.jsonl') else ''

In [ ]:
STEPS = 40000
!python schematic_ocr.py train --out schematic-ocr --steps {STEPS} --batch 128 {REAL}

In [ ]:
!test -f real/holdout.jsonl && python schematic_ocr.py eval --model schematic-ocr --data real/holdout.jsonl
!zip -qr schematic-ocr.zip schematic-ocr && echo done
files.download('schematic-ocr.zip')